# Automotive Field Failure Intelligence

## Project Objective

Analyze real-world NHTSA vehicle complaint data from 2020–2024 to identify recurring
vehicle and component complaint patterns, understand failure characteristics,
quantify safety-related severity, and identify areas that may warrant deeper
engineering investigation.

## Data Source

NHTSA Vehicle Safety Complaints — 2020–2024

- Records: 418,885
- Source: National Highway Traffic Safety Administration (NHTSA)
- Data type: Consumer/vehicle safety complaints
- Coverage: 2020–2024

> Note: NHTSA complaint records represent reported complaints and should not be
> interpreted as verified failure rates or confirmed manufacturing defects.

## Analytical Workflow

Raw NHTSA Complaint Data
        ↓
Data Profiling & Quality Assessment
        ↓
Data Cleaning & Transformation
        ↓
SQL-Based Complaint Analysis
        ↓
Python-Based Engineering & Text Analysis
        ↓
Power BI Decision Dashboard
        ↓
Engineering / Business Insights

## Key Questions

1. Which vehicle makes, models, and components generate the highest complaint volumes?
2. Which vehicle/model-year combinations show notable complaint concentrations?
3. At what mileage do different complaint patterns emerge?
4. Which complaints are associated with crashes, fires, injuries, or fatalities?
5. Which component × vehicle combinations warrant deeper engineering investigation?
6. What recurring failure themes can be identified from complaint narratives?

## 1. Data Profiling

Before cleaning or transforming the dataset, the first step is to assess:

- Dataset size and structure
- Date coverage
- Manufacturer and vehicle diversity
- Component diversity
- Safety-related indicators
- Missing and invalid values
- Mileage quality
- Complaint narrative availability

The profiling stage will be used to determine which fields and analytical
dimensions are reliable enough for downstream SQL, Python, and Power BI analysis.

In [1]:
import pandas as pd
import numpy as np


In [2]:
file_path = r"D:\Projects\SQL\Automotive Field Failure Intelligence\dataset\raw\NHTSA_Complaints_2020_2024_Raw.xlsx"

df = pd.read_excel(file_path)

print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]}")

Rows: 418,884
Columns: 49


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 418884 entries, 0 to 418883
Data columns (total 49 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   CMPLID             418884 non-null  int64  
 1   ODINO              418884 non-null  int64  
 2   MFR_NAME           418877 non-null  object 
 3   MAKETXT            418877 non-null  object 
 4   MODELTXT           418877 non-null  object 
 5   YEARTXT            418877 non-null  float64
 6   CRASH              418884 non-null  object 
 7   FAILDATE           418884 non-null  int64  
 8   FIRE               418884 non-null  object 
 9   INJURED            418884 non-null  int64  
 10  DEATHS             418884 non-null  int64  
 11  COMPDESC           418882 non-null  object 
 12  CITY               418867 non-null  object 
 13  STATE              418884 non-null  object 
 14  VIN                409447 non-null  object 
 15  DATEA              418884 non-null  int64  
 16  LD

In [4]:
df.head()

,CMPLID,ODINO,MFR_NAME,MAKETXT,MODELTXT,YEARTXT,CRASH,FAILDATE,FIRE,INJURED,...,RESTRAINT_TYPE,DEALER_NAME,DEALER_TEL,DEALER_CITY,DEALER_STATE,DEALER_ZIP,PROD_TYPE,REPAIRED_YN,MEDICAL_ATTN,VEHICLES_TOWED_YN
0,1633421,11292384,Honda (American Honda Motor Co.),HONDA,ACCORD,2018.0,N,20191221,N,0,...,NaN,NaN,NaN,NaN,NaN,NaN,V,NaN,N,N
1,1633422,11292384,Honda (American Honda Motor Co.),HONDA,ACCORD,2018.0,N,20191221,N,0,...,NaN,NaN,NaN,NaN,NaN,NaN,V,NaN,N,N
2,1633423,11292384,Honda (American Honda Motor Co.),HONDA,ACCORD,2018.0,N,20191221,N,0,...,NaN,NaN,NaN,NaN,NaN,NaN,V,NaN,N,N
3,1633424,11292385,Ford Motor Company,FORD,EXPLORER,2020.0,N,20191226,N,0,...,NaN,NaN,NaN,NaN,NaN,NaN,V,NaN,N,N
4,1633425,11292386,"General Motors, LLC",CHEVROLET,VOLT,2017.0,N,20190712,N,0,...,NaN,NaN,NaN,NaN,NaN,NaN,V,NaN,N,N


### Initial Dataset Structure

The dataset contains 418,884 complaint records across 49 fields covering
vehicle identification, incident characteristics, component information,
mileage, safety outcomes, complaint narratives, and vehicle attributes.

The next checks focus on identifying the usable analytical dimensions and
potential data-quality issues before transformation.

In [5]:
print("Dataset Shape:")
print(df.shape)

print("\nUnique Manufacturers:")
print(df["MFR_NAME"].nunique(dropna=True))

print("\nUnique Makes:")
print(df["MAKETXT"].nunique(dropna=True))

print("\nUnique Components:")
print(df["COMPDESC"].nunique(dropna=True))

Dataset Shape:
(418884, 49)

Unique Manufacturers:
611

Unique Makes:
715

Unique Components:
680


In [6]:
df["LDATE"] = pd.to_datetime(df["LDATE"].astype(str),format="%Y%m%d",errors="coerce")

print("Complaint Date Range:")
print("Earliest:", df["LDATE"].min())
print("Latest:  ", df["LDATE"].max())

Complaint Date Range:
Earliest: 2020-01-01 00:00:00
Latest:   2024-12-31 00:00:00


In [7]:
print("CRASH distribution:")
print(df["CRASH"].value_counts(dropna=False))

print("\nFIRE distribution:")
print(df["FIRE"].value_counts(dropna=False))

print("\nRecords with injuries > 0:")
print((pd.to_numeric(df["INJURED"], errors="coerce") > 0).sum())

print("\nRecords with deaths > 0:")
print((pd.to_numeric(df["DEATHS"], errors="coerce") > 0).sum())

CRASH distribution:
CRASH
N    399759
Y     19125
Name: count, dtype: int64

FIRE distribution:
FIRE
N    410869
Y      8015
Name: count, dtype: int64

Records with injuries > 0:
11417

Records with deaths > 0:
411


In [8]:
missing = (df.isna().sum().sort_values(ascending=False).to_frame("missing_count"))

missing["missing_pct"] = ( missing["missing_count"] / len(df) * 100).round(2)

missing.head(15)

,missing_count,missing_pct
FUEL_SYS,418593,99.93
TIRE_SIZE,418535,99.92
MANUF_DT,418295,99.86
RESTRAINT_TYPE,418268,99.85
SEAT_TYPE,418218,99.84
TIRE_FAIL_TYPE,418214,99.84
TRANS_TYPE,417274,99.62
ORIG_EQUIP_YN,417188,99.60
DOT,416692,99.48
PURCH_DT,416316,99.39


### Checking the Key Analytical Fields

The first missing-value check showed that several optional fields are rarely populated.
Before cleaning the data, we'll now focus on the fields that matter most to our analysis.

This check helps us understand whether vehicle details, mileage, component information,
and complaint descriptions are reliable enough to use in the next stages.

In [9]:
# Convert key numeric fields safely
numeric_cols = ["MILES", "INJURED", "DEATHS", "OCCURENCES", "VEH_SPEED", "NUM_CYLS"]

for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# Profile important analytical fields
profile_cols = [
    "MFR_NAME",
    "MAKETXT",
    "MODELTXT",
    "YEARTXT",
    "COMPDESC",
    "MILES",
    "INJURED",
    "DEATHS",
    "CDESCR"
]

profile = pd.DataFrame({
    "dtype": df[profile_cols].dtypes.astype(str),
    "missing_count": df[profile_cols].isna().sum(),
    "missing_pct": (df[profile_cols].isna().mean() * 100).round(2),
    "unique_values": df[profile_cols].nunique(dropna=True)
})

profile.sort_values("missing_pct", ascending=False)

,dtype,missing_count,missing_pct,unique_values
MILES,float64,268824,64.18,19138
CDESCR,object,25,0.01,289311
MFR_NAME,object,7,0.00,611
MAKETXT,object,7,0.00,715
MODELTXT,object,7,0.00,3484
YEARTXT,float64,7,0.00,55
COMPDESC,object,2,0.00,680
INJURED,int64,0,0.00,18
DEATHS,int64,0,0.00,10


In [10]:
print("MILES summary:")
print(df["MILES"].describe())

print("\nYEARTXT value counts:")
print(df["YEARTXT"].value_counts(dropna=False).head(15))

print("\nTop 15 components:")
print(df["COMPDESC"].value_counts(dropna=False).head(15))

MILES summary:
count    1.500600e+05
mean     7.690616e+04
std      1.087573e+05
min      0.000000e+00
25%      2.700000e+04
50%      6.800000e+04
75%      1.100000e+05
max      9.848609e+06
Name: MILES, dtype: float64

YEARTXT value counts:
YEARTXT
2018.0    39487
2017.0    38845
2016.0    35862
2019.0    34807
2015.0    30528
2014.0    28140
2013.0    26458
2020.0    26385
2021.0    24568
2022.0    22742
2023.0    17613
2012.0    16924
2011.0    13828
2010.0     9119
2008.0     7677
Name: count, dtype: int64

Top 15 components:
COMPDESC
ENGINE                                                      59770
ELECTRICAL SYSTEM                                           48954
UNKNOWN OR OTHER                                            41033
POWER TRAIN                                                 38004
STEERING                                                    28889
SERVICE BRAKES                                              25359
AIR BAGS                                                   

### Validating Mileage Quality

Mileage can help us understand when reported problems emerge during vehicle use,
but the field is incomplete and may contain unusual values.

We'll separate missing, zero, typical, and extreme mileage values before deciding
how to use mileage in the analysis.

In [11]:
miles = df["MILES"]

print("Total records:", len(miles))
print("Missing:", miles.isna().sum())
print("Zero mileage:", (miles == 0).sum())
print("Positive mileage:", (miles > 0).sum())
print("Above 500,000 miles:", (miles > 500_000).sum())
print("Above 1,000,000 miles:", (miles > 1_000_000).sum())

Total records: 418884
Missing: 268824
Zero mileage: 4552
Positive mileage: 145508
Above 500,000 miles: 337
Above 1,000,000 miles: 156


### Mileage Profiling Takeaway

Mileage is available for only a subset of complaints, so it will be treated as
an analytical dimension rather than a mandatory field.

The raw mileage values will be preserved. During transformation, we will create
meaningful mileage groups and separately investigate unusually high values.

### Checking Model Year

Model year will be an important dimension for comparing complaint patterns.
We'll check for missing or unusual year values before cleaning.

In [14]:
year = pd.to_numeric(df["YEARTXT"], errors="coerce")

print("Missing model year:", year.isna().sum())
print("Unknown / N/A (9999):", (year == 9999).sum())

print("\nEarliest model year:", year.min())
print("Latest model year:", year.max())

print("\nLowest model years:")
print(year.value_counts().sort_index().head(15))

print("\nHighest model years:")
print(year.value_counts().sort_index().tail(15))

Missing model year: 7
Unknown / N/A (9999): 5924

Earliest model year: 1960.0
Latest model year: 9999.0

Lowest model years:
YEARTXT
1960.0     1
1969.0     5
1970.0     1
1971.0     1
1974.0     3
1975.0     1
1977.0     2
1978.0     1
1979.0     1
1981.0    13
1982.0    34
1983.0     3
1984.0     3
1985.0     6
1986.0    10
Name: count, dtype: int64

Highest model years:
YEARTXT
2012.0    16924
2013.0    26458
2014.0    28140
2015.0    30528
2016.0    35862
2017.0    38845
2018.0    39487
2019.0    34807
2020.0    26385
2021.0    24568
2022.0    22742
2023.0    17613
2024.0     7023
2025.0      624
9999.0     5924
Name: count, dtype: int64


### Model Year Takeaway

Model year is mostly usable. The `9999` code represents unknown/N/A values,
so we'll convert it to missing during cleaning.

In [15]:
# Create a clean model-year field while preserving the raw column
df["MODEL_YEAR"] = pd.to_numeric(df["YEARTXT"], errors="coerce")

df.loc[df["MODEL_YEAR"] == 9999, "MODEL_YEAR"] = np.nan

print("Valid model year:", df["MODEL_YEAR"].notna().sum())
print("Missing / unknown:", df["MODEL_YEAR"].isna().sum())

Valid model year: 412953
Missing / unknown: 5931


### Checking Record Uniqueness

We'll verify the complaint identifier before creating the analytical dataset.

In [16]:
print("Total records:", len(df))
print("Unique CMPLID:", df["CMPLID"].nunique())
print("Duplicate CMPLID records:", df["CMPLID"].duplicated().sum())

Total records: 418884
Unique CMPLID: 418884
Duplicate CMPLID records: 0


### Profiling Complete

The dataset has 418,884 unique complaint records with usable vehicle,
component, safety, and date information. Key data-quality issues identified
during profiling will be handled during the preparation stage.

In [17]:
# Save the profiled dataset for SQL analysis
output_file = "NHTSA_Complaints_2020_2024_Profiled.csv"

df.to_csv(output_file, index=False)

print(f"Saved: {output_file}")
print(f"Rows: {len(df):,}")
print(f"Columns: {len(df.columns)}")

Saved: NHTSA_Complaints_2020_2024_Profiled.csv
Rows: 418,884
Columns: 50
